# Flood Extent Mapping of the Feni River (Sentinel-1 Radar)

This notebook maps the extent of the August 2024 flood in Feni District, Bangladesh,
using Sentinel-1 radar satellite images and change detection in Google Earth Engine.

Radar is used because heavy monsoon cloud made ordinary optical satellite images
unusable during the flood. Radar sends its own signal and can see through clouds.

**The steps below:** connect to Earth Engine, define the area and dates, load the
before and after radar images, detect what changed, clean the result with a few
filters, measure the flooded area, and finally save the maps as images.


## 1. Connect to Google Earth Engine

This project needs a free Google Earth Engine account and a project ID.
**Replace the project ID below with your own** before running.

In [ ]:
# Connect this session to Google Earth Engine.
# The first time it runs, a sign-in window opens. This is only needed once per session.
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='even-archway-473806-j5')   # <-- put your own project ID here
print("Connected to Earth Engine.")

## 2. Define the Study Area and Dates

The study area is Feni District. Its boundary is loaded automatically from the FAO GAUL
administrative dataset, so it is exact and anyone can reproduce it. Two date windows are
set: a "before" window in early August 2024, and an "after" window at the flood peak
in late August.

In [ ]:
# Get the Feni District boundary automatically from the FAO GAUL dataset.
aoi = ee.FeatureCollection("FAO/GAUL/2015/level2") \
        .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh')) \
        .filter(ee.Filter.eq('ADM2_NAME', 'Feni')) \
        .geometry()

# Work out the real ground area of the district, in square kilometres.
area_of_aoi = aoi.area(maxError=1).getInfo() / 1e6
print("Study area (Feni District):", round(area_of_aoi, 2), "square km")

# Date windows for the August 2024 flood: before the flood, and at the flood peak.
before_start, before_end = '2024-08-01', '2024-08-15'
after_start,  after_end  = '2024-08-22', '2024-08-31'
print("Date windows set.")

## 3. Load the Sentinel-1 Radar Images

Sentinel-1 collects radar images of the whole planet. We keep only the images that use
the VV signal type over land, and that cover Feni. For each period (before and after) we
build one median image, which reduces random radar noise.

In [ ]:
# Load the Sentinel-1 radar image collection and keep the images we need.
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.bounds(aoi))

# Build one median radar image for each period, cut to the district boundary.
before = s1.select('VV').filterDate(before_start, before_end).median().clip(aoi)
after  = s1.select('VV').filterDate(after_start,  after_end ).median().clip(aoi)

print("Radar images ready: 'before' and 'after'.")

## 4. Detect What Changed

We subtract the before image from the after image to see what changed.

In this rural, crop-heavy floodplain, floodwater sitting among plants makes the radar
signal **brighter**, not darker (a "double-bounce" effect between the water and the
vegetation). So areas that got brighter after the flood are the areas of interest.
In the map below, red means brighter (possible flooding) and blue means darker.

In [ ]:
# Subtract before from after. Positive values = the area got brighter.
difference = after.subtract(before)

# Show the change on an interactive map (red = brighter, blue = darker).
diff_vis = {'min': -5, 'max': 5, 'palette': ['blue', 'white', 'red']}
change_map = geemap.Map()
change_map.centerObject(aoi, 12)
change_map.addLayer(difference, diff_vis, 'Change (after - before)')
change_map

## 5. Flag and Clean the Flooded Pixels

Raw change detection catches a lot of false alarms, so three filters clean it up:

1. **Brightness threshold** keeps only pixels that got clearly brighter.
2. **Slope mask** removes steep hillsides, because water does not pool on slopes.
3. **Permanent-water mask** removes the existing river channel, so we do not count
   water that was already there as new flooding.

In [ ]:
# 1. Keep pixels that got clearly brighter after the flood.
raw_flood = difference.gt(3)

# 2. Slope filter: keep only flat land (slope under 5 degrees), where water pools.
dem = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(dem).clip(aoi)
slope_mask = slope.lt(5)

# 3. Permanent-water filter: in VV radar, open water is usually below -16 dB.
#    Keeping pixels that were NOT already water removes the existing river channel.
permanent_water_mask = before.gte(-16)

# Combine all three filters into the final, cleaned flood layer.
flood_cleaned = raw_flood.updateMask(slope_mask).updateMask(permanent_water_mask)
flood_only    = flood_cleaned.updateMask(flood_cleaned)   # hide non-flood pixels for a clean map

# Show the cleaned flood (red) on top of the before radar image.
flood_map = geemap.Map()
flood_map.centerObject(aoi, 11)
flood_map.addLayer(before, {'min': -25, 'max': 0}, 'Before flood (radar)')
flood_map.addLayer(flood_only, {'palette': ['red']}, 'Cleaned flood extent')
flood_map.add_layer_control()
flood_map

## 6. Measure the Flooded Area

Each flood pixel is turned into its real ground area, then summed, to get the total
flooded area in square kilometres and as a share of the district.

In [ ]:
# Multiply each flood pixel by its real ground area, then add them all up.
pixel_area = flood_cleaned.multiply(ee.Image.pixelArea())
flood_area = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
).getInfo()

area_sqkm = flood_area['VV'] / 1e6
percent_flooded = area_sqkm / area_of_aoi * 100

print("District area:", round(area_of_aoi, 2), "square km")
print("Flooded area: ", round(area_sqkm, 2), "square km")
print("Share flooded:", round(percent_flooded, 1), "%")

## 7. Save the Maps as Images

Earth Engine maps are interactive, so they show as blank boxes on GitHub. This step
saves each map as a plain PNG image that displays anywhere, with no sign-in needed by
the viewer. A 5 km buffer is added around the district so the edges are not cut off.

In [ ]:
import urllib.request
from PIL import Image, ImageDraw, ImageFont

# Build the coloured version of each map.
radar_grey = before.visualize(bands=['VV'], min=-25, max=0)          # radar in greyscale
flood_red  = flood_only.visualize(palette=['red'])                   # flood pixels in red
flood_over_radar = radar_grey.blend(flood_red)                       # red flood on grey radar

change_coloured = difference.visualize(min=-5, max=5,
                                       palette=['blue', 'white', 'red'])

before_grey = before.visualize(bands=['VV'], min=-25, max=0)
after_grey  = after.visualize(bands=['VV'],  min=-25, max=0)

# Add a 5 km buffer so the district edges have some breathing room.
export_region = aoi.buffer(5000).bounds()

def save_png(coloured_map, filename, dimensions=1024):
    """Ask Earth Engine for a PNG picture of a map and download it."""
    url = coloured_map.getThumbURL({
        'region': export_region,
        'dimensions': dimensions,
        'format': 'png'
    })
    urllib.request.urlretrieve(url, filename)
    print("saved", filename)

save_png(flood_over_radar, "flood_extent_map.png")
save_png(change_coloured,  "change_map.png")
save_png(before_grey,      "radar_before.png")
save_png(after_grey,       "radar_after.png")

# Put 'before' and 'after' side by side in one labelled image.
def add_label(image_path, text):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, img.width, 50], fill=(0, 0, 0))   # black title bar
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 24)
    except IOError:
        font = ImageFont.load_default()
    draw.text((15, 13), text, fill=(255, 255, 255), font=font)   # y=13 centres the text
    return img

left  = add_label("radar_before.png", "Before (early August)")
right = add_label("radar_after.png",  "After (flood peak)")

gap = 12
combined = Image.new("RGB",
                     (left.width + right.width + gap, max(left.height, right.height)),
                     (255, 255, 255))
combined.paste(left, (0, 0))
combined.paste(right, (left.width + gap, 0))
combined.save("before_after.png")
print("saved before_after.png")

## 8. Locator Map

This saves a clean map of Bangladesh with Feni filled in red, so viewers can see where
the study area is. It is exported as a plain base map; the title, the "FENI" arrow, and
the country label were added afterwards in an image editor.

In [ ]:
import urllib.request

# Country outline and the Feni district shape.
bd = ee.FeatureCollection("FAO/GAUL/2015/level0") \
        .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))
feni = ee.FeatureCollection("FAO/GAUL/2015/level2") \
        .filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh')) \
        .filter(ee.Filter.eq('ADM2_NAME', 'Feni'))

# Draw Bangladesh as a grey outline and fill Feni in red.
bd_outline = ee.Image().paint(bd, 1, 2)   # country border, 2 pixels wide
feni_fill  = ee.Image().paint(feni, 1)    # Feni solid
locator = bd_outline.visualize(palette=['444444']) \
            .blend(feni_fill.visualize(palette=['red']))

# 50 km buffer so the northern tips of the country are not cut off.
buffered_region = bd.geometry().buffer(50000).bounds()

url = locator.getThumbURL({
    'region': buffered_region,
    'dimensions': 800,
    'format': 'png'
})
urllib.request.urlretrieve(url, "locator_map.png")
print("saved locator_map.png (clean base map; add labels in an image editor)")